In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
generator_llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash-lite', max_retries=5)
evaluator_llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash-lite', max_retries=5)
optimizer_llm = ChatGoogleGenerativeAI(model='gemini-3.6-flash-lite', max_retries=5)

In [3]:
from typing import TypedDict, Annotated, Literal
import operator

class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal['approved', 'needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

def generate_tweet(state: TweetState):
    messages = [
        SystemMessage(content='You are a funny and clever Twitter influencer.'),
        HumanMessage(content=f'''Write a short, original, and hilarious tweet on the topic: {state['topic']}.

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day-to-day English.''')
    ]

    response = generator_llm.invoke(messages).content

    return {'tweet': response, 'tweet_history': [response]}

In [6]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal['approved', 'needs_improvement'] = Field(description='Final evaluation result')
    feedback: str = Field(description='Feedback for the tweet')

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)


def evaluate_tweet(state: TweetState):
    messages = [
        SystemMessage(content='You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format.'),
        HumanMessage(content=f'''Evaluate the following tweet:

Tweet: {state['tweet']}

Use the criteria below:
1. Originality - Is this fresh, or have you seen it 100 times before?
2. Humor - Did it genuinely make you smile or laugh?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A format)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did X? Because Y").
- It exceeds 280 characters.
- It uses hashtags or emojis.
- It's a traditional setup-punchline joke.

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining strengths and weaknesses.''')
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}



In [7]:
def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content='You punch up tweets for virality and humor based on given feedback.'),
        HumanMessage(content=f'''Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.''')
    ]

    response = optimizer_llm.invoke(messages).content

    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [8]:
def route_evaluation(state: TweetState) -> Literal['approved', 'needs_improvement']:
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [9]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})

graph.add_edge('optimize', 'evaluate')  

workflow = graph.compile()